In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
requests.get("https://www.ambitionbox.com/list-of-companies?page=1").text 
# permission denied as data is in robots.txt 

'<HTML><HEAD>\n<TITLE>Access Denied</TITLE>\n</HEAD><BODY>\n<H1>Access Denied</H1>\n \nYou don\'t have permission to access "http&#58;&#47;&#47;www&#46;ambitionbox&#46;com&#47;list&#45;of&#45;companies&#63;" on this server.<P>\nReference&#32;&#35;18&#46;57882c31&#46;1786895264&#46;7221367b\n<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;18&#46;57882c31&#46;1786895264&#46;7221367b</P>\n</BODY>\n</HTML>\n'

## if code is 403 (permission denied)
1. ask you browser "what is my user agent"
2. get agent url mention it into header variable during request with prefix "User-Agent":

this disguies your auttomated http request to normal browser request

## Requesting data from single WebPage

In [1]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:153.0) Gecko/20100101 Firefox/153.0"
}
webpage = requests.get("https://www.ambitionbox.com/list-of-companies?page=1", headers=headers).text

# this returns the html content of webpage

In [4]:
soup = BeautifulSoup(webpage, 'lxml') # parser and stores the structured data

In [2]:
"""print(soup.prettify())"""#clean nd properly indented structure

'print(soup.prettify())'

## cross cheking/ identifing data from stored data or fetched data 

In [6]:
soup.find_all('h1')[0].text  # returns txt inside the header h1 

'\n\t\t\t\t\tTop Companies in\n\t\t\t\t \n\t\t\t\t\tINDIA\n\t\t\t\t\t'

In [7]:
for i in soup.find_all('h2'):  # returns text of all secondary headers h2 
    print(i.text.strip())      # stip remove unnecssary spacing

Companies in India
TCS
Accenture
Wipro
Cognizant
Capgemini
HDFC Bank
Infosys
HCLTech
ICICI Bank
Tech Mahindra
Genpact
TP
Axis Bank
Jio
Concentrix Corporation
Amazon
Reliance Retail
iEnergizer
LTM Limited
HDB Financial Services
Popular Collections by Industries
Popular Collections by Cities
Popular Collections by Roles


In [8]:
company = soup.find_all('div',class_='companyCardWrapper')  #extracting the block of all company  which present in div class companyCardWrapper

In [9]:
len(company)

20

In [10]:
# returns all content instide that specific div tag 
company[1].find_all('div',class_='companyCardWrapper__tertiaryInformation')[0].text.strip()

'76.2k Reviews7.3L Salaries9.6k Interviews14.5k Jobs7k Benefits49 Photos'

## Scraping the required and essential data 

In [11]:
# extracting the required data from each block using loop , and mentioning the tag (h2 div span) to address the required data

# creating a empty lists
name = []
rating = []
review = []
salary = []
interviews = []
jobs = []
benefits = []

# looping for all companies
for i in company:
    name.append(i.find('h2', class_='companyCardWrapper__companyName').text.strip())  # finds company name which is in h2 tag in class mentioned

    rating.append(i.find('div', class_='rating_text').text.strip())   # extracting rating of each comapny by indexing 

    data = i.find_all('span', class_='companyCardWrapper__ActionCount')
    # as salaries reviews etc are inside same tag span, we stored whole tag inside list data,
    # we extract each by indexing it from data

    review.append(data[0].text.strip())  # as first element in list is Reviews its index is 0
    salary.append(data[1].text.strip())
    interviews.append(data[2].text.strip())
    jobs.append(data[3].text.strip())
    benefits.append(data[4].text.strip())



In [12]:
#combining all lists into one dictionary
d = {
    'name': name,
    'rating': rating,
    'reviews': review,
    'salary': salary,
    'interviews': interviews,
    'jobs': jobs,
    'benefits': benefits
}

In [13]:
# converting it to a DataFrame
df = pd.DataFrame(d)

In [14]:
df.shape

(20, 7)

In [15]:
df

,name,rating,reviews,salary,interviews,jobs,benefits
0,TCS,3.3,1.2L,10.4L,11.4k,5k,11k
1,Accenture,3.7,76.2k,7.3L,9.6k,14.5k,7k
2,Wipro,3.6,67.1k,4.9L,7k,2,4.9k
3,Cognizant,3.7,63.4k,6.1L,6.6k,873,5.7k
4,Capgemini,3.6,55.6k,5L,5.7k,2.1k,3.9k
5,HDFC Bank,3.8,54.7k,1.5L,3.2k,529,3.4k
6,Infosys,3.5,50.5k,5.3L,8.6k,3.4k,4.9k
7,HCLTech,3.4,48.3k,4L,4.7k,280,4k
8,ICICI Bank,4.0,47.2k,1.6L,3k,17,3.8k
9,Tech Mahindra,3.3,44.8k,2.9L,4.7k,779,3.5k


## For multiple pages 

In [16]:
final = pd.DataFrame()

for j in range(1,55):
    url = "https://www.ambitionbox.com/list-of-companies?page={}".format(j)
    headers = {"User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:153.0) Gecko/20100101 Firefox/153.0"}
    webpage = requests.get(url, headers=headers).text

    soup = BeautifulSoup(webpage, 'lxml')

    company = soup.find_all('div',class_='companyCardWrapper') 
    
    name = []
    rating = []
    review = []
    salary = []
    interviews = []
    jobs = []
    benefits = []

    for i in company:
        name.append(i.find('h2', class_='companyCardWrapper__companyName').text.strip()) 
    
        rating.append(i.find('div', class_='rating_text').text.strip()) 
    
        data = i.find_all('span', class_='companyCardWrapper__ActionCount')
        
        review.append(data[0].text.strip())  
        salary.append(data[1].text.strip())
        interviews.append(data[2].text.strip())
        jobs.append(data[3].text.strip())
        benefits.append(data[4].text.strip())

        d = {
        'name': name,
        'rating': rating,
        'reviews': review,
        'salary': salary,
        'interviews': interviews,
        'jobs': jobs,
        'benefits': benefits}

    df = pd.DataFrame(d)
    
    final = pd.concat([final, df], ignore_index=True)
    

In [17]:
final.shape

(1080, 7)

In [18]:
final

,name,rating,reviews,salary,interviews,jobs,benefits
0,TCS,3.3,1.2L,10.4L,11.4k,5k,11k
1,Accenture,3.7,76.2k,7.3L,9.6k,14.5k,7k
2,Wipro,3.6,67.1k,4.9L,7k,2,4.9k
3,Cognizant,3.7,63.4k,6.1L,6.6k,873,5.7k
4,Capgemini,3.6,55.6k,5L,5.7k,2.1k,3.9k
...,...,...,...,...,...,...,...
1075,AstraZeneca,3.8,1k,7.5k,71,1,151
1076,Sahara India Pariwar,3.9,1k,2.5k,31,--,89
1077,National Payments Corporation of India,3.9,1k,4.5k,107,148,41
1078,Amazon Web Services,3.8,1k,7.1k,196,--,84


## Working with dataframe 

In [19]:
final.iloc[29]

name          IDFC FIRST Bank
rating                    4.0
reviews                 15.7k
salary                  51.6k
interviews               1.1k
jobs                       42
benefits                  825
Name: 29, dtype: str

In [20]:
final.iloc[329]

name          Micro Labs
rating               3.6
reviews             2.8k
salary             10.1k
interviews           158
jobs                  19
benefits             208
Name: 329, dtype: str

In [21]:
final.iloc[699]

name          QualityKiosk Technologies
rating                              3.2
reviews                            1.5k
salary                            11.8k
interviews                          168
jobs                                222
benefits                            102
Name: 699, dtype: str

In [28]:
final[final['name'].str.contains('Tata', case=False, na=False)]

,name,rating,reviews,salary,interviews,jobs,benefits
33,Tata Motors,4.1,14600.0,44400.0,1300.0,23.0,1700.0
46,Tata Steel,4.0,10300.0,34900.0,1100.0,NaN,693.0
84,Tata Projects,4.2,7100.0,23100.0,653.0,60.0,591.0
114,Tata Capital,4.0,5900.0,16900.0,304.0,5200.0,282.0
160,Tata AIA Life Insurance,3.8,4800.0,15300.0,271.0,873.0,318.0
192,Tata Communications,3.8,4100.0,23900.0,313.0,260.0,431.0
262,Tata Technologies,3.4,3400.0,23800.0,320.0,318.0,323.0
263,Tata Electronics,3.8,3400.0,14300.0,307.0,6.0,126.0
264,Tata AIG,3.9,3400.0,12100.0,203.0,61.0,244.0
265,Tata AutoComp Systems,3.6,3400.0,11900.0,228.0,114.0,194.0


In [22]:
final['rating'].dtype

<StringDtype(na_value=nan)>

In [23]:
final['rating'] = pd.to_numeric(final['rating'])   # type convertionfrom string to double

In [24]:
final[final['rating'] >4.8]

,name,rating,reviews,salary,interviews,jobs,benefits
294,Marpu Foundation,4.9,3.1k,61,266,--,8
496,Tekwissen,4.9,2k,628,341,277,140
753,Hire In Global,4.9,1.4k,8,304,--,36
817,Royal Migration Solutions,4.9,1.4k,5,1,--,1


In [25]:
## To clean the data and convert columns to appropriate data types
import re

def parse_indian_number(val):
    if pd.isna(val):
        return None
    val = str(val).strip().upper()
    match = re.match(r'^([\d.]+)([KL]?)$', val)
    if not match:
        return None
    num, suffix = match.groups()
    num = float(num)
    if suffix == 'K':
        num *= 1_000
    elif suffix == 'L':
        num *= 100_000
    return num

cols_to_convert = ['reviews', 'salary', 'interviews', 'jobs', 'benefits']
for col in cols_to_convert:
    final[col] = final[col].apply(parse_indian_number)

final['rating'] = pd.to_numeric(final['rating'], errors='coerce')

print(final.dtypes)
final

name              str
rating        float64
reviews       float64
salary        float64
interviews    float64
jobs          float64
benefits      float64
dtype: object


,name,rating,reviews,salary,interviews,jobs,benefits
0,TCS,3.3,120000.0,1040000.0,11400.0,5000.0,11000.0
1,Accenture,3.7,76200.0,730000.0,9600.0,14500.0,7000.0
2,Wipro,3.6,67100.0,490000.0,7000.0,2.0,4900.0
3,Cognizant,3.7,63400.0,610000.0,6600.0,873.0,5700.0
4,Capgemini,3.6,55600.0,500000.0,5700.0,2100.0,3900.0
...,...,...,...,...,...,...,...
1075,AstraZeneca,3.8,1000.0,7500.0,71.0,1.0,151.0
1076,Sahara India Pariwar,3.9,1000.0,2500.0,31.0,NaN,89.0
1077,National Payments Corporation of India,3.9,1000.0,4500.0,107.0,148.0,41.0
1078,Amazon Web Services,3.8,1000.0,7100.0,196.0,NaN,84.0


## Downloading the file in csv format

In [26]:
#final.to_csv('Companies.csv', index=False)
# 

In [4]:
final[final['rating'] >4.]

NameError: name 'final' is not defined